# Exploration ML (CatBoost)

Dataset d'entrainement, observations hotel, exports LOO.


## 1. Setup


In [ ]:
from pathlib import Path
import sys

# Racine release (dossier de ce notebook)
ROOT = Path.cwd().resolve()
if not (ROOT / "pipeline").is_dir():
    # si le kernel a un autre cwd, remonter depuis le fichier
    ROOT = Path(".").resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.pipeline.paths import Paths, release_root
from src.pipeline.connection import PipelineFactory
from src.pipeline.engine import ConnectionPipeline

paths = Paths(ROOT).ensure()
print("ROOT     :", paths.root)
print("DB       :", paths.main_db, "exists=", paths.main_db.exists())
print("pipeline :", paths.pipeline)


## 2. Connexion


In [ ]:
# Connexion lecture/ecriture sur la base principale + YAML pipeline/
cp = PipelineFactory(paths).open(read_only=False)
print("project_dir :", cp.project_dir)
print("objets pipeline YAML :", len(cp.pipeline))


## 3. Relations ML / dataset


In [ ]:
import duckdb

def list_relations(cp, like: str | None = None):
    """Liste tables et vues de la base ouverte."""
    q = """
        SELECT table_name, table_type
        FROM information_schema.tables
        WHERE table_schema = current_schema()
        ORDER BY table_type, table_name
    """
    df = cp.con.execute(q).df()
    if like:
        df = df[df["table_name"].str.contains(like, case=False, na=False)]
    return df

rels = list_relations(cp)
display(rels)
print(f"{len(rels)} relations")

ml_rels = list_relations(cp, like=r"ml|dataset_pivot|sales")
display(ml_rels)


## 4. Vue `v_ml_training_dataset` (prerequis auto)


In [ ]:
try:
    ds = cp.p_table_view("v_ml_training_dataset").df()
    display(ds.head(20))
    print("shape", ds.shape)
    print("hotels", ds["hotel_code"].nunique() if "hotel_code" in ds.columns else None)
    print("cols", list(ds.columns)[:30], "...")
except Exception as exc:
    print("v_ml_training_dataset:", exc)


## 5. Observations reelles (une ligne par hotel)


In [ ]:
try:
    obs = cp.con.execute("""
        SELECT *
        FROM v_ml_training_dataset
        WHERE is_observation
        ORDER BY hotel_code
    """).df()
    display(obs)
except Exception as exc:
    print(exc)


## 6. Service CatBoost — metriques LOO si Excel present


In [ ]:
from src.ml.catboost_model import CatBoostService
from pathlib import Path

svc = CatBoostService(paths)
excel = paths.out_ml("eval_catboost_loo.xlsx")
print("excel LOO :", excel, "exists=", excel.exists())

if excel.exists():
    import pandas as pd
    metrics = pd.read_excel(excel, sheet_name="metrics")
    preds = pd.read_excel(excel, sheet_name="predictions")
    display(metrics)
    display(preds.head(20))
else:
    print("Pas encore d'eval — lancer: python run.py ml --rebuild")


## 7. Modeles sauvegardes


In [ ]:
models_dir = paths.models_catboost
files = sorted(models_dir.glob("*")) if models_dir.exists() else []
for f in files:
    print(f.name, f.stat().st_size)
if not files:
    print("(aucun modele .cbm)")


## 8. Apercu dataset (stats simples)


In [ ]:
try:
    ds = cp.p_table_view("v_ml_training_dataset").df()
    targets = [
        "montant_ventes_par_mois",
        "montant_marge_par_mois",
        "montant_marge_selon_coef_par_mois",
    ]
    cols = [c for c in targets if c in ds.columns]
    display(ds[cols].describe())
    if "solution" in ds.columns:
        display(ds.groupby("solution").size().rename("n_lignes"))
except Exception as exc:
    print(exc)


## 9. Fermer


In [ ]:
cp.close()
print("connexion fermee")
